# 12.3 Node embeddings: DeepWalk and node2vec

Random walks make graph neighborhoods look like sentences.

The transition matrix $P=D^{-1}A$ turns edges into walk probabilities. Walk context windows then create co-occurrence counts that can be compressed by SVD or learned embedding objectives.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

SEED = 12
rng = np.random.default_rng(SEED)
random.seed(SEED)


def adjacency_from_graph(G):
    nodes = list(G.nodes())
    return nx.to_numpy_array(G, nodelist=nodes, dtype=float)


def labels_from_graph(G):
    labels = []
    for node in G.nodes():
        club = G.nodes[node].get("club", "Mr. Hi")
        value = 1 if club == "Officer" else 0
        labels.append(value)
    return np.array(labels, dtype=int)


def degree_features(A):
    degree = A.sum(axis=1)
    max_degree = max(1.0, float(degree.max()))
    normalized_degree = degree / max_degree
    clustering = np.array([nx.clustering(nx.from_numpy_array(A), i) for i in range(A.shape[0])])
    return np.column_stack([normalized_degree, clustering])


def make_sbm_graph(sizes, p_in, p_out, seed, feature_noise):
    blocks = len(sizes)
    probs = np.full((blocks, blocks), p_out, dtype=float)
    np.fill_diagonal(probs, p_in)
    G = nx.stochastic_block_model(sizes, probs, seed=seed)
    y = []
    for block_id, size in enumerate(sizes):
        y.extend([block_id] * size)
    y = np.array(y, dtype=int)
    base = np.eye(blocks, dtype=float)[y]
    local_rng = np.random.default_rng(seed)
    noise = local_rng.normal(0.0, feature_noise, size=base.shape)
    X = base + noise
    return G, X, y


def make_graph_ladder(seed=SEED):
    ladder = []

    G1 = nx.Graph()
    G1.add_edges_from([(0, 1), (0, 2), (1, 2), (1, 3)])
    X1 = np.array([[1.0, 0.0], [1.0, 1.0], [0.0, 1.0], [0.0, 0.0]])
    y1 = np.array([0, 0, 1, 1], dtype=int)
    ladder.append({"rung": "D1", "name": "4-node toy", "G": G1, "A": adjacency_from_graph(G1), "X": X1, "y": y1})

    G2 = nx.karate_club_graph()
    A2 = adjacency_from_graph(G2)
    X2 = degree_features(A2)
    y2 = labels_from_graph(G2)
    ladder.append({"rung": "D2", "name": "karate club", "G": G2, "A": A2, "X": X2, "y": y2})

    G3, X3, y3 = make_sbm_graph([18, 18], 0.34, 0.04, seed + 3, 0.28)
    ladder.append({"rung": "D3", "name": "synthetic SBM with noise", "G": G3, "A": adjacency_from_graph(G3), "X": X3, "y": y3})

    G4, X4, y4 = make_sbm_graph([30, 30, 30], 0.28, 0.07, seed + 4, 0.35)
    ladder.append({"rung": "D4", "name": "larger denser SBM", "G": G4, "A": adjacency_from_graph(G4), "X": X4, "y": y4})

    G5, X5, y5 = make_sbm_graph([35, 35, 35], 0.18, 0.11, seed + 5, 0.55)
    hub_edges = []
    for node in range(10, 95, 7):
        hub_edges.append((0, node))
    G5.add_edges_from(hub_edges)
    A5 = adjacency_from_graph(G5)
    ladder.append({"rung": "D5", "name": "noisy hub-heavy SBM", "G": G5, "A": A5, "X": X5, "y": y5})

    return ladder


def centroid_predict(Z, y):
    classes = np.unique(y)
    centroids = []
    for cls in classes:
        centroids.append(Z[y == cls].mean(axis=0))
    centroids = np.vstack(centroids)
    distances = ((Z[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    indices = distances.argmin(axis=1)
    return classes[indices]


def node_accuracy(pred, y):
    return float(np.mean(np.asarray(pred) == np.asarray(y)))


def conductance(A, labels):
    labels = np.asarray(labels)
    group = labels == labels[0]
    if group.all() or (not group.any()):
        return 1.0
    cut = float(A[np.ix_(group, ~group)].sum())
    volume_a = float(A[group].sum())
    volume_b = float(A[~group].sum())
    denom = max(1e-12, min(volume_a, volume_b))
    return cut / denom


def normalized_adjacency(A):
    A_tilde = A + np.eye(A.shape[0])
    degree = A_tilde.sum(axis=1)
    inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(degree, 1e-12)))
    return inv_sqrt @ A_tilde @ inv_sqrt


def plot_graph_panels(results, metric_name):
    cols = len(results)
    fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 3))
    if cols == 1:
        axes = [axes]
    for ax, item in zip(axes, results):
        G = item["G"]
        colors = item["pred"]
        pos = nx.spring_layout(G, seed=SEED)
        nx.draw_networkx_nodes(G, pos, node_color=colors, cmap="viridis", node_size=80, ax=ax)
        nx.draw_networkx_edges(G, pos, alpha=0.25, width=0.7, ax=ax)
        ax.set_title(f'{item["rung"]}: {item["metric"]:.3f}')
        ax.axis("off")
    plt.show()

    plt.figure(figsize=(6, 3))
    xs = [item["rung"] for item in results]
    ys = [item["metric"] for item in results]
    plt.plot(xs, ys, marker="o")
    plt.ylabel(metric_name)
    plt.xlabel("ladder rung")
    plt.title(f"{metric_name} vs rung")
    plt.grid(True, alpha=0.3)
    plt.show()


## 3. The concept, built once (D1)

DeepWalk builds $P=D^{-1}A$, samples deterministic walk windows here, counts co-occurrences $C$, and embeds nodes with a low-rank factorization. The lesson checks are transition row $[0.5,0,0.5,0]$, $(P^2)_{0,2}=0.500$, $C_{1,0}=5$, $C_{1,2}=2$, and node2vec return/outward probabilities $[0.800,0.200]$ for $p=0.5,q=2$.

In [ ]:

def walk_embedding(A, dim=2):
    degree = A.sum(axis=1)
    P = np.divide(A, degree[:, None], out=np.zeros_like(A), where=degree[:, None] > 0)
    walks = [[1, 0, 1, 0, 1, 0], [1, 2, 1], [0, 1, 2, 3]]
    C = np.zeros_like(A)
    for walk in walks:
        for position, center in enumerate(walk):
            lo = max(0, position - 1)
            hi = min(len(walk), position + 2)
            for context_position in range(lo, hi):
                if context_position != position:
                    context = walk[context_position]
                    C[center, context] += 1
    U, S, Vt = np.linalg.svd(C + P + P @ P, full_matrices=False)
    Z = U[:, :dim] * np.sqrt(S[:dim])
    return P, C, Z

def node2vec_transition_weights(A, previous, current, p=0.5, q=2.0):
    candidates = np.where(A[current] > 0)[0]
    weights = []
    for candidate in candidates:
        if candidate == previous:
            weights.append(1.0 / p)
        elif A[previous, candidate] > 0:
            weights.append(1.0)
        else:
            weights.append(1.0 / q)
    weights = np.array(weights, dtype=float)
    probs = weights / weights.sum()
    return candidates, probs

G_path = nx.path_graph(4)
A_path = adjacency_from_graph(G_path)
P_path, C_path, Z_path = walk_embedding(A_path)
candidates_path, probs_path = node2vec_transition_weights(A_path, 0, 1, p=0.5, q=2.0)

assert np.allclose(P_path[1], [0.5, 0.0, 0.5, 0.0])
assert round(float((P_path @ P_path)[0, 2]), 3) == 0.500
assert int(C_path[1, 0]) == 5
assert int(C_path[1, 2]) == 2
assert np.allclose(np.round(probs_path, 3), [0.800, 0.200])


The embedding is a compressed record of where walks carry context. Node2vec changes that record by reweighting return, in-neighborhood, and outward steps before counts are formed.

In [ ]:

print("transition row for node 1", P_path[1].tolist())
print("two-step probability 0 to 2", round(float((P_path @ P_path)[0, 2]), 3))
print("co-occurrence row 1", C_path[1].astype(int).tolist())
print("node2vec candidates", candidates_path.tolist())
print("node2vec probabilities", np.round(probs_path, 3).tolist())


## 4. The dataset ladder

The F11 graph ladder is inline and CPU-only: D1 is a hand-checkable four-node toy, D2 is the bundled NetworkX karate club graph, D3 is a noisy stochastic block model, D4 is a larger/denser synthetic graph, and D5 is a harder noisy hub-heavy graph.

In [ ]:

ladder = make_graph_ladder()

for item in ladder:
    A = item["A"]
    X = item["X"]
    y = item["y"]
    edge_count = int(A.sum() // 2)
    classes = np.unique(y).tolist()
    print(item["rung"], item["name"])
    print("  nodes", A.shape[0], "edges", edge_count, "features", X.shape, "classes", classes)
    print("  sample X", np.round(X[:3], 3).tolist())


In [ ]:

results = []

for item in ladder:
    P, C, Z_walk = walk_embedding(item["A"], dim=min(3, len(np.unique(item["y"]))))
    Z = np.column_stack([Z_walk, item["X"][:, :1]])
    pred = centroid_predict(Z, item["y"])
    metric = node_accuracy(pred, item["y"])
    results.append({"rung": item["rung"], "G": item["G"], "pred": pred, "metric": metric})
    print(f'{item["rung"]} {item["name"]:24s} node-accuracy={metric:.3f}')


In [ ]:

plot_graph_panels(results, "node accuracy")


## 7. Pitfall on the hardest rung: disconnected components cannot share walk evidence

If a component is isolated, no context window crosses into it. Report component-aware metrics or add legitimate bridging evidence rather than pretending the walk sampler saw missing paths.

In [ ]:

hard = ladder[-1]
G_aug = hard["G"].copy()
offset = G_aug.number_of_nodes()
G_aug.add_edges_from([(offset, offset + 1), (offset + 1, offset + 2)])
components = list(nx.connected_components(G_aug))
component_sizes = sorted([len(component) for component in components], reverse=True)
can_cross = nx.has_path(G_aug, 0, offset)

print("component sizes", component_sizes[:5])
print("walk context can cross from main graph to new component", can_cross)
print("fix: train and report within connected components or add observed bridge edges")


## 8. Evaluate it + Practice

- Report the ladder metric (node accuracy) next to a no-skill majority-class or random partition baseline.
- Sanity-check D1 against the hand-computed assertions before trusting larger rungs.
- Ablation: set the walk window to zero so co-occurrence disappears.
- Failure signals: good embeddings on connected rungs but arbitrary vectors on isolated components.
- Keep all experiments CPU-only and seeded.

Practice prompts:
1. Change D3 noise and predict which metric moves first.
2. Add a small bridge between communities and inspect the visualization.
3. Replace the readout with a majority baseline and compare the curve.